# Models

Three gradient-boosted models — the right family for tabular. `TUNE = True` re-runs Optuna and caches `best_params.json`; otherwise each model loads its tuned params and just produces out-of-fold predictions.

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import pandas as pd
from pathlib import Path
import optuna
from sklearn.metrics import roc_auc_score
from src.data import load_selected
from src.models import oof

optuna.logging.set_verbosity(optuna.logging.WARNING)
TUNE = False
INTERIM = Path("../data/interim")
X, y = load_selected()
bp = json.loads((INTERIM / "best_params.json").read_text()) if (INTERIM / "best_params.json").exists() else {}
preds = {}

def cv(model):
    return roc_auc_score(y, oof(model, X, y))

def tune(name, objective, n_trials):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    bp[name] = study.best_params
    (INTERIM / "best_params.json").write_text(json.dumps(bp, indent=2))
    print(f"  best CV AUC {study.best_value:.4f}")

def evalm(name, model):
    preds[name] = oof(model, X, y)
    print(f"{name}: OOF AUC {roc_auc_score(y, preds[name]):.4f}")

X.shape

## LightGBM

In [ ]:
import lightgbm as lgb
def obj_lgb(t):
    return cv(lgb.LGBMClassifier(
        n_estimators=t.suggest_int("n_estimators", 300, 1200),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=t.suggest_int("num_leaves", 16, 255),
        min_child_samples=t.suggest_int("min_child_samples", 20, 300),
        subsample=t.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
        colsample_bytree=t.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        reg_alpha=t.suggest_float("reg_alpha", 1e-3, 20.0, log=True),
        n_jobs=-1, verbose=-1))
if TUNE: tune("lgb", obj_lgb, 20)
p = {"n_estimators": 800, "learning_rate": 0.03, "num_leaves": 31, "min_child_samples": 50,
     "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 2.0, **bp.get("lgb", {})}
evalm("lgb", lgb.LGBMClassifier(**p, subsample_freq=1, n_jobs=-1, verbose=-1))

## XGBoost

In [ ]:
import xgboost as xgb
def obj_xgb(t):
    return cv(xgb.XGBClassifier(tree_method="hist", eval_metric="auc", n_jobs=-1,
        n_estimators=t.suggest_int("n_estimators", 300, 1200),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth=t.suggest_int("max_depth", 3, 10),
        min_child_weight=t.suggest_int("min_child_weight", 1, 20),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        reg_alpha=t.suggest_float("reg_alpha", 1e-3, 20.0, log=True)))
if TUNE: tune("xgb", obj_xgb, 20)
p = {"n_estimators": 800, "max_depth": 6, "learning_rate": 0.03, "subsample": 0.8,
     "colsample_bytree": 0.7, "reg_lambda": 2.0, **bp.get("xgb", {})}
evalm("xgb", xgb.XGBClassifier(**p, tree_method="hist", eval_metric="auc", n_jobs=-1))

## CatBoost

In [ ]:
from catboost import CatBoostClassifier
def obj_cat(t):
    return cv(CatBoostClassifier(verbose=0, allow_writing_files=False, thread_count=-1,
        iterations=t.suggest_int("iterations", 300, 1200),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.1, log=True),
        depth=t.suggest_int("depth", 4, 10),
        l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        random_strength=t.suggest_float("random_strength", 1e-3, 10.0, log=True)))
if TUNE: tune("cat", obj_cat, 12)
p = {"iterations": 800, "learning_rate": 0.03, "depth": 6, "l2_leaf_reg": 3, **bp.get("cat", {})}
evalm("cat", CatBoostClassifier(**p, verbose=0, allow_writing_files=False))

# Seed bag

In [ ]:
lgb_p = {"n_estimators": 800, "learning_rate": 0.03, "num_leaves": 31, "min_child_samples": 50,
         "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 2.0, **bp.get("lgb", {})}
seeds = {s: oof(lgb.LGBMClassifier(**lgb_p, subsample_freq=1, random_state=s, n_jobs=-1, verbose=-1), X, y, seed=s)
         for s in range(10)}
preds["lgb_bag"] = pd.DataFrame(seeds).mean(axis=1).to_numpy()
print(f"lgb single {roc_auc_score(y, preds['lgb']):.4f}   lgb 10-seed bag {roc_auc_score(y, preds['lgb_bag']):.4f}")

# Save

In [ ]:
oofs = pd.DataFrame(preds, index=X.index)
oofs.to_pickle(INTERIM / "predictions.pkl")
oofs.corr().round(3)